In [ ]:
# %%
# =============================================================================
# NOTEBOOK: kSZ × v_rec Cross-Power Spectrum
# =============================================================================
#
# PURPOSE
# -------
# Measure C_ell(T_kSZ × v_rec) in redshift bins z=5–8 and compare to
# C_ell(T_kSZ × v_true) — the reconstruction penalty.
#
# WHAT THIS NEEDS FROM YOUR MAIN NOTEBOOK
# ----------------------------------------
# Run the save cell (CELL EXPORT below) in your main notebook first.
# It saves the arrays this notebook loads.
#
# PIPELINE
# ---------
# 1. Load: kSZ integrand slices, true velocity, LAE catalogue, geometry
# 2. Per z-bin:
#    a. Paint LAE number density onto grid → delta_LAE
#    b. FFT linear reconstruction → v_rec (from delta_LAE)
#    c. Weight kSZ integrand by v_rec → momentum field q_rec
#    d. Weight kSZ integrand by v_true → momentum field q_true
#    e. 2D FFT → C_ell(q_rec) and C_ell(q_true) for this z-bin
# 3. Plot: C_ell vs ell per z-bin, and C_ell at fixed ell vs z
# =============================================================================

# %%
# =============================================================================
# CELL EXPORT  — paste this at the END of your main notebook (kSZ_sqr_halo_Jun4)
# Run it once to save the arrays this notebook needs.
# =============================================================================

# import numpy as np
# import os
#
# EXPORT_DIR = "ksz_vrec_inputs"
# os.makedirs(EXPORT_DIR, exist_ok=True)
#
# for seed in RANDOM_SEEDS:
#     sd = os.path.join(EXPORT_DIR, f"seed_{seed}")
#     os.makedirs(sd, exist_ok=True)
#
#     # kSZ integrand: shape (HII_DIM, HII_DIM, n_z)
#     np.save(f"{sd}/ksz_integrand.npy", kSZ_integrand_all[seed].astype(np.float32))
#
#     # True LOS velocity: shape (HII_DIM, HII_DIM, n_lc)
#     np.save(f"{sd}/v_true.npy", los_velocity_lc_all[seed].astype(np.float32))
#
#     # LAE catalogue
#     lae_data = load_seed(seed)
#     if lae_data is not None:
#         np.save(f"{sd}/lae_coords.npy",    lae_data["coords"])      # (N,3) cMpc
#         np.save(f"{sd}/lae_redshifts.npy", lae_data["redshifts"])   # (N,)
#         np.save(f"{sd}/lae_mask.npy",      lae_data["is_LAE"])      # (N,) bool
#
# # Geometry — same for all seeds
# np.save(f"{EXPORT_DIR}/z_lc.npy",  z_lc)           # (n_lc,) lightcone redshifts
# np.save(f"{EXPORT_DIR}/z_mid.npy", tau_results_all[RANDOM_SEEDS[0]]['z_mid'])
# np.save(f"{EXPORT_DIR}/geometry.npy",
#         np.array([BOX_LEN, HII_DIM], dtype=np.float64))
#
# print("Export complete →", EXPORT_DIR)

# %%
# =============================================================================
# CELL 1: Imports and config
# =============================================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from astropy.cosmology import FlatLambdaCDM

# ── paths ──────────────────────────────────────────────────────────────────────
EXPORT_DIR  = "ksz_vrec_inputs"          # written by CELL EXPORT above
PLOT_DIR    = "ksz_vrec_plots"
RANDOM_SEEDS = list(range(1, 6))

os.makedirs(PLOT_DIR, exist_ok=True)

# ── cosmology ──────────────────────────────────────────────────────────────────
cosmo  = FlatLambdaCDM(H0=67.77, Om0=0.3086, Ob0=0.0489)
h      = 0.6777
f_grow = 0.97    # linear growth rate f ≈ Omega_m(z)^0.55, roughly constant ~0.97 at z~7

# ── redshift bins for the analysis ────────────────────────────────────────────
# coarse enough to have LAEs, fine enough to track reionization
Z_BINS = [(5.0, 5.5), (5.5, 6.0), (6.0, 6.5), (6.5, 7.0), (7.0, 7.5), (7.5, 8.0)]

# smoothing scale for reconstruction [cMpc] — matches typical bubble scale
R_SMOOTH_MPC = 10.0

# ── plot settings ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'serif', 'mathtext.fontset': 'cm',
    'font.size': 14, 'axes.labelsize': 14,
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.top': True, 'ytick.right': True,
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

print("✓ Cell 1 done")

# %%
# =============================================================================
# CELL 2: Load geometry and arrays
# =============================================================================

geom    = np.load(f"{EXPORT_DIR}/geometry.npy")
BOX_LEN = float(geom[0])   # cMpc
HII_DIM = int(geom[1])
cell_Mpc = BOX_LEN / HII_DIM

z_lc    = np.load(f"{EXPORT_DIR}/z_lc.npy")    # (n_lc,)
z_mid   = np.load(f"{EXPORT_DIR}/z_mid.npy")   # (n_z-1,)  — midpoints of integrand

print(f"✓ BOX_LEN={BOX_LEN} cMpc  HII_DIM={HII_DIM}  cell={cell_Mpc:.2f} cMpc")
print(f"  z_lc range: {z_lc.min():.2f} – {z_lc.max():.2f}  ({len(z_lc)} slices)")

# ── load per-seed arrays ───────────────────────────────────────────────────────
ksz_int_all  = {}   # kSZ integrand (HII_DIM, HII_DIM, n_z)
v_true_all   = {}   # true velocity  (HII_DIM, HII_DIM, n_lc)
lae_all      = {}   # dict with coords, redshifts, is_LAE

for seed in RANDOM_SEEDS:
    sd = os.path.join(EXPORT_DIR, f"seed_{seed}")
    if not os.path.exists(sd):
        print(f"  ✗ seed {seed}: export dir missing — run CELL EXPORT first")
        continue

    ksz_int_all[seed] = np.load(f"{sd}/ksz_integrand.npy")   # float32 fine
    v_true_all [seed] = np.load(f"{sd}/v_true.npy")

    coords    = np.load(f"{sd}/lae_coords.npy")
    redshifts = np.load(f"{sd}/lae_redshifts.npy")
    is_lae    = np.load(f"{sd}/lae_mask.npy")

    lae_all[seed] = {
        "coords":    coords,
        "redshifts": redshifts,
        "is_LAE":    is_lae,
    }

    n_lae = is_lae.sum()
    print(f"  ✓ seed {seed}:  integrand={ksz_int_all[seed].shape}  "
          f"v_true={v_true_all[seed].shape}  LAEs={n_lae:,}")

print("✓ Cell 2 done")

# %%
# =============================================================================
# CELL 3: Helper functions
# =============================================================================

def paint_lae_density(coords_lae, HII_DIM, BOX_LEN):
    """
    Paint LAE positions onto a 2D grid using cloud-in-cell (CIC).
    Returns delta_LAE = (N_grid / N_mean) - 1, shape (HII_DIM, HII_DIM).
    Only uses x,y positions — projection along z (LOS).

    Parameters
    ----------
    coords_lae : (N, 3) array, comoving [cMpc]

    Returns
    -------
    delta_lae : (HII_DIM, HII_DIM) overdensity field
    """
    cell = BOX_LEN / HII_DIM
    grid = np.zeros((HII_DIM, HII_DIM), dtype=np.float64)

    xi = np.clip((coords_lae[:, 0] / cell).astype(int), 0, HII_DIM - 1)
    yi = np.clip((coords_lae[:, 1] / cell).astype(int), 0, HII_DIM - 1)
    np.add.at(grid, (xi, yi), 1.0)

    mean_n = grid.mean()
    if mean_n > 0:
        delta = grid / mean_n - 1.0
    else:
        delta = grid  # empty bin — return zeros

    return delta.astype(np.float32)


def reconstruct_velocity(delta_lae, BOX_LEN, z_bin_center,
                         f_grow=0.97, R_smooth=10.0, cosmo=None):
    """
    Linear velocity reconstruction from a 2D LAE density field.
    Solves the 2D continuity equation in Fourier space:
        v_rec(k) = i * aHf * k / k^2 * delta_lae(k) * W_G(k)

    This is the 2D projected version — appropriate for a redshift slice.

    Parameters
    ----------
    delta_lae    : (HII_DIM, HII_DIM) LAE overdensity
    BOX_LEN      : box side [cMpc]
    z_bin_center : central redshift of the bin
    f_grow       : linear growth rate
    R_smooth     : Gaussian smoothing scale [cMpc]
    cosmo        : astropy cosmology

    Returns
    -------
    v_rec_los : (HII_DIM, HII_DIM) reconstructed LOS velocity [Mpc/s]
    """
    from astropy import units as u

    N    = delta_lae.shape[0]
    dk   = 2.0 * np.pi / BOX_LEN                     # fundamental mode [1/cMpc]
    kx   = np.fft.fftfreq(N, d=1.0/N) * dk           # [1/cMpc]
    ky   = np.fft.fftfreq(N, d=1.0/N) * dk
    KX, KY = np.meshgrid(kx, ky, indexing='ij')
    K2   = KX**2 + KY**2
    K2[0, 0] = 1.0   # avoid divide by zero at k=0

    # Gaussian smoothing kernel
    K = np.sqrt(K2)
    W_G = np.exp(-0.5 * (K * R_smooth)**2)

    # aHf in Mpc/s  (velocity units matching v_true)
    a   = 1.0 / (1.0 + z_bin_center)
    H_z = cosmo.H(z_bin_center).to(u.km / u.s / u.Mpc).value * 1e3 / 3.0857e22  # s^-1
    aHf = a * H_z * f_grow * 3.0857e22   # Mpc/s per (1/Mpc) = Mpc^2/s — wait
    # cleaner: keep in km/s then convert
    H_km_s_Mpc = cosmo.H(z_bin_center).to(u.km / u.s / u.Mpc).value
    aHf_km_s   = a * H_km_s_Mpc * f_grow  # km/s / (Mpc^-1) units

    # FFT of density
    delta_k = np.fft.fft2(delta_lae)

    # LOS = z direction, but we only have 2D projected map.
    # Use KX as the LOS proxy (matches the x-direction velocity convention
    # in 21cmFAST where velocity_z is along the LOS = z-axis of the box).
    # If your LOS is along a different axis, change KX → KY below.
    v_rec_k = (1j * KX / K2) * W_G * delta_k * aHf_km_s

    # inverse FFT → real space
    v_rec = np.real(np.fft.ifft2(v_rec_k)).astype(np.float32)

    return v_rec   # units: km/s  (same order as v_true in Mpc/s × 1e-3)


def power_spectrum_2d(field1, field2, BOX_LEN, n_ell_bins=20):
    """
    Cross-power spectrum of two 2D fields via FFT.
    Returns ell, C_ell, and D_ell = ell*(ell+1)/(2pi) * C_ell.

    Parameters
    ----------
    field1, field2 : (N, N) real arrays — must be same shape
    BOX_LEN        : physical side length [cMpc]
    n_ell_bins     : number of annular bins in k-space

    Returns
    -------
    ell    : (n_ell_bins,) multipole values
    C_ell  : (n_ell_bins,) cross-power [(field units)^2 cMpc^2]
    D_ell  : (n_ell_bins,) ell*(ell+1)/(2pi) * C_ell
    n_modes: (n_ell_bins,) number of modes per bin
    """
    N    = field1.shape[0]
    dk   = 2.0 * np.pi / BOX_LEN
    kx   = np.fft.fftfreq(N, d=1.0/N) * dk
    ky   = np.fft.fftfreq(N, d=1.0/N) * dk
    KX, KY = np.meshgrid(kx, ky, indexing='ij')
    K    = np.sqrt(KX**2 + KY**2)

    # convert k [1/cMpc] → ell via flat-sky: ell = k * chi
    # for a projected 2D map we just use k directly and label axis as k [h/Mpc]
    # ell bins: log-spaced from fundamental to Nyquist
    k_min = dk
    k_max = np.sqrt(2) * (N // 2) * dk
    k_edges = np.logspace(np.log10(k_min), np.log10(k_max), n_ell_bins + 1)

    F1 = np.fft.fft2(field1)
    F2 = np.fft.fft2(field2)
    cross = np.real(F1 * np.conj(F2)) * (BOX_LEN / N)**2  # power normalization

    C_ell   = np.zeros(n_ell_bins)
    n_modes = np.zeros(n_ell_bins, dtype=int)
    k_cents = 0.5 * (k_edges[:-1] + k_edges[1:])

    for i in range(n_ell_bins):
        mask = (K >= k_edges[i]) & (K < k_edges[i+1])
        n_modes[i] = mask.sum()
        if n_modes[i] > 0:
            C_ell[i] = cross[mask].mean() / BOX_LEN**2

    D_ell = k_cents * (k_cents + 1) / (2 * np.pi) * C_ell

    return k_cents, C_ell, D_ell, n_modes


print("✓ Cell 3: helper functions defined")

# %%
# =============================================================================
# CELL 4: Main loop — compute C_ell per z-bin per seed
# =============================================================================

print("\n" + "="*70)
print("CELL 4 — C_ell(T_kSZ × v_rec) and C_ell(T_kSZ × v_true) per z-bin")
print("="*70)

# results storage
# cross_results[seed][z_label] = {'k', 'C_rec', 'C_true', 'D_rec', 'D_true', 'n_modes'}
cross_results = {seed: {} for seed in RANDOM_SEEDS}

for seed in RANDOM_SEEDS:
    if seed not in ksz_int_all:
        continue

    print(f"\n── seed {seed} ──")

    ksz_int  = ksz_int_all[seed]    # (HII_DIM, HII_DIM, n_z)
    v_true   = v_true_all[seed]     # (HII_DIM, HII_DIM, n_lc)
    lae_data = lae_all[seed]

    coords_all    = lae_data["coords"]
    redshifts_all = lae_data["redshifts"]
    is_lae        = lae_data["is_LAE"]

    # only LAEs
    coords_lae    = coords_all[is_lae]
    redshifts_lae = redshifts_all[is_lae]

    for (z_lo, z_hi) in Z_BINS:
        z_center = 0.5 * (z_lo + z_hi)
        z_label  = f"z{z_lo:.1f}-{z_hi:.1f}"

        # ── select LAEs in this z-bin ──────────────────────────────────────
        zmask = (redshifts_lae >= z_lo) & (redshifts_lae < z_hi)
        n_lae_bin = zmask.sum()

        if n_lae_bin < 10:
            print(f"  {z_label}: only {n_lae_bin} LAEs — skipping")
            continue

        # ── select kSZ integrand slices in this z-bin ─────────────────────
        # z_mid has the redshift of each integrand slice
        # integrand shape is (HII_DIM, HII_DIM, n_z_mid)
        n_z_mid = ksz_int.shape[2]
        z_mid_use = z_mid[:n_z_mid] if len(z_mid) >= n_z_mid else z_mid
        z_slice_mask = (z_mid_use >= z_lo) & (z_mid_use < z_hi)
        n_slices = z_slice_mask.sum()

        if n_slices == 0:
            print(f"  {z_label}: no integrand slices — skipping")
            continue

        # ── select v_true slices in this z-bin ────────────────────────────
        v_slice_mask = (z_lc >= z_lo) & (z_lc < z_hi)

        # ── step 1: LAE density field → v_rec ─────────────────────────────
        delta_lae = paint_lae_density(coords_lae[zmask], HII_DIM, BOX_LEN)
        v_rec_2d  = reconstruct_velocity(
            delta_lae, BOX_LEN, z_center,
            f_grow=f_grow, R_smooth=R_SMOOTH_MPC, cosmo=cosmo
        )   # (HII_DIM, HII_DIM) in km/s

        # ── step 2: average kSZ integrand and v_true over z-bin slices ────
        ksz_zbin   = ksz_int[:, :, z_slice_mask].mean(axis=2)   # (HII_DIM, HII_DIM)

        if v_slice_mask.sum() > 0:
            v_true_2d = v_true[:, :, v_slice_mask].mean(axis=2)  # Mpc/s
            # convert to km/s to match v_rec units
            v_true_2d_km = v_true_2d * 3.0857e19 / 1e3           # Mpc/s → km/s
        else:
            v_true_2d_km = np.zeros((HII_DIM, HII_DIM), dtype=np.float32)
            print(f"  {z_label}: no v_true slices found")

        # ── step 3: momentum fields ────────────────────────────────────────
        q_rec  = ksz_zbin * v_rec_2d      # T_kSZ × v_rec
        q_true = ksz_zbin * v_true_2d_km  # T_kSZ × v_true

        # ── step 4: cross-power spectra ────────────────────────────────────
        k, C_rec,  D_rec,  n_m = power_spectrum_2d(q_rec,  q_rec,  BOX_LEN)
        _,  C_true, D_true, _   = power_spectrum_2d(q_true, q_true, BOX_LEN)
        _,  C_cross, D_cross, _ = power_spectrum_2d(q_rec,  q_true, BOX_LEN)

        cross_results[seed][z_label] = {
            'k':       k,
            'C_rec':   C_rec,
            'C_true':  C_true,
            'C_cross': C_cross,
            'D_rec':   D_rec,
            'D_true':  D_true,
            'D_cross': D_cross,
            'n_modes': n_m,
            'n_lae':   int(n_lae_bin),
            'n_slices': int(n_slices),
            'z_center': z_center,
        }

        # reconstruction correlation coefficient r
        with np.errstate(invalid='ignore', divide='ignore'):
            r = C_cross / np.sqrt(np.abs(C_rec * C_true))
        r_median = float(np.nanmedian(r[n_m > 5]))

        print(f"  {z_label}: {n_lae_bin:,} LAEs  {n_slices} slices  "
              f"r_median={r_median:.3f}")

print("\n✓ Cell 4 done")

# %%
# =============================================================================
# CELL 5: Seed-averaged results
# =============================================================================

print("\n" + "="*70)
print("CELL 5 — Seed-averaged C_ell")
print("="*70)

# collect all z_labels that appear in at least one seed
all_z_labels = set()
for seed in RANDOM_SEEDS:
    all_z_labels.update(cross_results[seed].keys())
all_z_labels = sorted(all_z_labels)

# averaged results
avg_results = {}

for z_label in all_z_labels:
    C_rec_list   = []
    C_true_list  = []
    D_rec_list   = []
    D_true_list  = []
    r_list       = []
    k_ref        = None

    for seed in RANDOM_SEEDS:
        if z_label not in cross_results[seed]:
            continue
        res = cross_results[seed][z_label]
        C_rec_list.append(res['C_rec'])
        C_true_list.append(res['C_true'])
        D_rec_list.append(res['D_rec'])
        D_true_list.append(res['D_true'])
        with np.errstate(invalid='ignore', divide='ignore'):
            r_list.append(res['C_cross'] / np.sqrt(np.abs(res['C_rec'] * res['C_true'])))
        if k_ref is None:
            k_ref = res['k']

    if len(C_rec_list) == 0:
        continue

    avg_results[z_label] = {
        'k':          k_ref,
        'C_rec_mean': np.nanmean(C_rec_list, axis=0),
        'C_rec_std':  np.nanstd (C_rec_list, axis=0),
        'C_true_mean':np.nanmean(C_true_list, axis=0),
        'C_true_std': np.nanstd (C_true_list, axis=0),
        'D_rec_mean': np.nanmean(D_rec_list, axis=0),
        'D_true_mean':np.nanmean(D_true_list, axis=0),
        'r_mean':     np.nanmean(r_list, axis=0),
        'r_std':      np.nanstd (r_list, axis=0),
        'z_center':   cross_results[
            next(s for s in RANDOM_SEEDS if z_label in cross_results[s])
        ][z_label]['z_center'],
        'n_seeds':    len(C_rec_list),
    }
    print(f"  {z_label}: averaged over {len(C_rec_list)} seeds")

print("✓ Cell 5 done")

# %%
# =============================================================================
# CELL 6: PLOT 1 — C_ell vs k per z-bin (rec vs true)
# =============================================================================

print("\n=== PLOT 1: C_ell vs k per z-bin ===")

cmap   = plt.cm.plasma
z_ctrs = [avg_results[zl]['z_center'] for zl in all_z_labels if zl in avg_results]
norm   = mpl.colors.Normalize(vmin=5.0, vmax=8.0)

fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
axes = axes.flatten()

for ax, z_label in zip(axes, all_z_labels):
    if z_label not in avg_results:
        ax.set_visible(False)
        continue

    res     = avg_results[z_label]
    k       = res['k']
    color   = cmap(norm(res['z_center']))

    # auto-power of true momentum field — upper bound
    ax.plot(k, np.abs(res['D_true_mean']),
            color=color, lw=2.5, ls='-',
            label=r'$v_{\rm true}$ (ceiling)')

    # auto-power of reconstructed momentum field
    ax.plot(k, np.abs(res['D_rec_mean']),
            color=color, lw=2.0, ls='--',
            label=r'$v_{\rm rec}$ (LAE recon)')

    ax.set_xscale('log')
    ax.set_yscale('symlog', linthresh=1e-20)
    ax.set_xlabel(r'$k$  [cMpc$^{-1}$]')
    ax.set_ylabel(r'$D_\ell \equiv \frac{k(k+1)}{2\pi} C_k$')
    ax.set_title(rf'$z = {res["z_center"]:.2f}$  ({res["n_seeds"]} seeds)',
                 fontsize=12)
    ax.legend(fontsize=10, loc='best')

for ax in axes[len(all_z_labels):]:
    ax.set_visible(False)

fig.suptitle(r'$C_k(T_{\rm kSZ} \times v)$ per redshift bin  —  rec vs true',
             fontsize=15, fontweight='bold')
fig.savefig(f"{PLOT_DIR}/C_ell_vs_k_per_zbin.png")
fig.savefig(f"{PLOT_DIR}/C_ell_vs_k_per_zbin.pdf")
plt.close(fig)
print("  ✓ C_ell_vs_k_per_zbin.png / .pdf")

# %%
# =============================================================================
# CELL 7: PLOT 2 — Reconstruction correlation r vs k per z-bin
# =============================================================================

print("\n=== PLOT 2: r(k) per z-bin ===")

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for z_label in all_z_labels:
    if z_label not in avg_results:
        continue
    res   = avg_results[z_label]
    color = cmap(norm(res['z_center']))
    valid = res['r_mean'] > -1.5   # remove pathological bins

    ax.plot(res['k'][valid], res['r_mean'][valid],
            color=color, lw=2.0, marker='o', markersize=4,
            label=rf"$z={res['z_center']:.2f}$")
    ax.fill_between(
        res['k'][valid],
        (res['r_mean'] - res['r_std'])[valid],
        (res['r_mean'] + res['r_std'])[valid],
        color=color, alpha=0.15
    )

ax.axhline(0, color='black', ls='--', lw=0.8, alpha=0.5)
ax.axhline(1, color='gray',  ls=':',  lw=0.8, alpha=0.5)
ax.set_xscale('log')
ax.set_ylim(-0.3, 1.2)
ax.set_xlabel(r'$k$  [cMpc$^{-1}$]')
ax.set_ylabel(r'$r(k) = C_k^{\rm cross} / \sqrt{C_k^{\rm rec}\,C_k^{\rm true}}$')
ax.set_title('Reconstruction correlation coefficient', fontweight='bold')

sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Redshift $z$', pad=0.02)

fig.savefig(f"{PLOT_DIR}/r_vs_k_per_zbin.png")
fig.savefig(f"{PLOT_DIR}/r_vs_k_per_zbin.pdf")
plt.close(fig)
print("  ✓ r_vs_k_per_zbin.png / .pdf")

# %%
# =============================================================================
# CELL 8: PLOT 3 — C_ell at fixed k vs redshift  (reionization history view)
# =============================================================================

print("\n=== PLOT 3: C_ell(z) at fixed k ===")

# pick representative k values
k_targets = [0.02, 0.05, 0.1]    # cMpc^-1 — large, medium, small scale
colors_k  = ['darkblue', 'darkgreen', 'darkred']

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

for ax, quantity, ylabel in zip(
    axes,
    ['D_true_mean', 'D_rec_mean'],
    [r'$D_k(T_{\rm kSZ}\times v_{\rm true})$',
     r'$D_k(T_{\rm kSZ}\times v_{\rm rec})$']
):
    for k_target, col in zip(k_targets, colors_k):
        z_pts, D_pts = [], []
        for z_label in all_z_labels:
            if z_label not in avg_results:
                continue
            res = avg_results[z_label]
            idx = int(np.argmin(np.abs(res['k'] - k_target)))
            D_val = res[quantity][idx]
            if np.isfinite(D_val):
                z_pts.append(res['z_center'])
                D_pts.append(np.abs(D_val))

        if len(z_pts) > 1:
            ax.plot(z_pts, D_pts, color=col, lw=2.5, marker='o', markersize=6,
                    label=rf'$k={k_target}$ cMpc$^{{-1}}$')

    ax.axhline(0, color='black', ls='--', lw=0.8, alpha=0.5)
    ax.set_xlabel(r'Redshift $z$')
    ax.set_ylabel(ylabel)
    ax.set_yscale('symlog', linthresh=1e-22)
    ax.invert_xaxis()
    ax.legend(fontsize=11)

axes[0].set_title('True velocity — upper bound', fontweight='bold')
axes[1].set_title('Reconstructed from LAEs', fontweight='bold')

fig.suptitle(r'$D_k$ vs redshift at fixed $k$  —  reionization history view',
             fontsize=14, fontweight='bold')

fig.savefig(f"{PLOT_DIR}/C_ell_vs_z_fixed_k.png")
fig.savefig(f"{PLOT_DIR}/C_ell_vs_z_fixed_k.pdf")
plt.close(fig)
print("  ✓ C_ell_vs_z_fixed_k.png / .pdf")

# %%
# =============================================================================
# CELL 9: Summary table — r per z-bin (reconstruction penalty)
# =============================================================================

print("\n" + "="*70)
print("SUMMARY: Reconstruction penalty (median r per z-bin)")
print("="*70)
print(f"{'z-bin':<18}  {'z_center':>8}  {'r_median':>10}  {'r_std':>8}  {'n_seeds':>8}")
print("-"*60)

for z_label in all_z_labels:
    if z_label not in avg_results:
        continue
    res      = avg_results[z_label]
    r_med    = float(np.nanmedian(res['r_mean']))
    r_std    = float(np.nanmedian(res['r_std']))
    print(f"{z_label:<18}  {res['z_center']:>8.2f}  "
          f"{r_med:>10.3f}  {r_std:>8.3f}  {res['n_seeds']:>8d}")

print("="*70)
print("\nr = 1.0 → perfect reconstruction  |  r = 0 → no signal recovered")
print("SNR(realistic) / SNR(true) ≈ r²")
print("="*70)